# 🤖 Agentic AI System Evaluation: Ragas vs DeepEval with Google Gemini

Welcome to the comprehensive masterclass on **Agentic AI System Evaluation** using **Ragas** and **DeepEval** powered by **Google Gemini**.

---

### 🎯 Learning Objectives:
1. **Understand Agentic System Evaluation Paradigms**: Learn how evaluating autonomous, tool-calling agents differs fundamentally from standard static RAG evaluation.
2. **Master the 3 Core Pillars of Agentic Evaluation**:
   - **Tool Selection Accuracy**: Did the agent select the exact right tool for the sub-task?
   - **Argument Correctness / Fidelity**: Did the agent correctly extract and format parameter values from user intent?
   - **Goal Attainment**: Did the cumulative reasoning steps and final answer satisfy the user's objective?
3. **Build an End-to-End Travel Planning Agent**: Construct a dynamic multi-tool LangChain agent powered by **Google Gemini** (`gemini-3.1-flash-lite`).
4. **Implement Ragas Agentic Evaluation Harness**:
   - Convert LangGraph agent states into Ragas message traces via `convert_to_ragas_messages`.
   - Measure `ToolCallAccuracy` (rule-based matching).
   - Measure `ToolCallF1` (precision/recall balance).
   - Measure `AgentGoalAccuracyWithReference` using an impartial Evaluator LLM (`gemini-3-flash-preview`).
5. **Implement DeepEval Agentic Evaluation Harness**:
   - Model multi-turn dialogues into formal `Turn` and `ConversationalTestCase` contracts.
   - Measure `ToolCorrectnessMetric` and `ArgumentCorrectnessMetric`.
   - Measure `GoalAccuracyMetric` using DeepEval's native `GeminiModel` (`gemini-3-flash-preview`).
6. **Comparative Analysis & Trade-Offs**: Compare Ragas and DeepEval evaluation philosophies, reporting formats, and production CI/CD suitability.

---

### 🔗 Official Documentation & References:
* **Ragas Documentation**: [https://docs.ragas.io/](https://docs.ragas.io/)
* **DeepEval Documentation**: [https://deepeval.com/](https://deepeval.com/)
* **LangChain Agents**: [https://python.langchain.com/docs/modules/agents/](https://python.langchain.com/docs/modules/agents/)
* **Google GenAI SDK**: [https://github.com/googleapis/python-genai](https://github.com/googleapis/python-genai)

## 1. Agentic AI Evaluation Taxonomy & Core Concepts

Evaluating autonomous agents presents unique engineering challenges compared to standard RAG pipelines:

| Evaluation Dimension | Traditional RAG Evaluation | Agentic AI System Evaluation |
| :--- | :--- | :--- |
| **System Workflow** | Deterministic: Retrieve $\to$ Augment $\to$ Generate | Non-deterministic: Reason $\to$ Act $\to$ Observe $\to$ Repeat |
| **Output Space** | Single final answer string | Multi-step execution trajectory + tool calls + final answer |
| **Failure Modes** | Hallucination, retrieval omission | Tool hallucination, infinite loops, bad arguments, goal drift |
| **Evaluation Primitives**| Groundedness, relevance, context recall | Tool precision, argument validation, goal attainment |

### 1.1 The 3 Evaluation Pillars for Agents:
1. **Tool Call Accuracy (Did it choose the right tool?)**:
   Verifies whether the agent invoked the expected function (e.g., `search_flights` instead of `get_weather` for a flight query).
2. **Argument Correctness (Did it pass the right parameters?)**:
   Verifies whether extracted arguments match expected criteria (e.g., origin="Bangalore", destination="Dubai", stars=4).
3. **Goal Accuracy (Did it solve the user's problem?)**:
   An LLM-as-a-Judge assessment determining if the cumulative dialogue trajectory fulfilled the user's stated goal.

## 2. End-to-End System Architecture

```text
What we are building and evaluating in this Notebook:

                               ┌────────────────────────────────┐
                               │       User Travel Query        │
                               └───────────────┬────────────────┘
                                               │
                                               ▼
                              ┌───────────────────────────────────┐
                              │  LangChain Travel Planning Agent  │
                              │ (ChatGoogleGenerativeAI)          │
                              │ model: gemini-3.1-flash-lite      │
                              └────────────────┬──────────────────┘
                                               │
                                ┌──────────────┴──────────────┐
                                ▼                             ▼
                    ┌───────────────────────┐     ┌───────────────────────┐
                    │   search_flights()    │     │     find_hotels()     │
                    └───────────────────────┘     └───────────────────────┘
                                │                             │
                                └──────────────┬──────────────┘
                                               │
                                               ▼
                                ┌───────────────────────────────┐
                                │   Agent Final Plan Response   │
                                └──────────────┬────────────────┘
                                               │
                       ┌───────────────────────┴───────────────────────┐
                       │                                               │
                       ▼                                               ▼
     ┌───────────────────────────────────┐           ┌───────────────────────────────────┐
     │    TRACK 1: RAGAS AGENT EVAL      │           │   TRACK 2: DEEPEVAL AGENT EVAL    │
     ├───────────────────────────────────┤           ├───────────────────────────────────┤
     │ Trace Adapter:                    │           │ Conversation Adapter:             │
     │ convert_to_ragas_messages()       │           │ ConversationalTestCase(turns)     │
     ├───────────────────────────────────┤           ├───────────────────────────────────┤
     │ Metrics:                          │           │ Metrics:                          │
     │ 1. ToolCallAccuracy (exact match) │           │ 1. ToolCorrectnessMetric          │
     │ 2. ToolCallF1 (precision/recall)  │           │ 2. ArgumentCorrectnessMetric      │
     │ 3. AgentGoalAccuracyWithReference │           │ 3. GoalAccuracyMetric             │
     │    (Judge: gemini-3-flash-preview)│           │    (Judge: gemini-3-flash-preview)│
     ├───────────────────────────────────┤           ├───────────────────────────────────┤
     │ Output:                           │           │ Output:                           │
     │ ragas_agent_evaluation_results.csv│           │ deepeval_agent_eval_results.csv   │
     └───────────────────────────────────┘           └───────────────────────────────────┘
```

---
## 3. Environment Setup & API Configuration

We import the required modules across LangChain, Google GenAI, Ragas, and DeepEval, and verify `GOOGLE_API_KEY`.

In [1]:
import os
import asyncio

import pandas as pd

from dotenv import load_dotenv

from google import genai

from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI

from ragas.llms import llm_factory

from ragas.messages import ToolCall

from ragas.metrics.collections import (
    ToolCallAccuracy,
    ToolCallF1,
    AgentGoalAccuracyWithReference,
)

from langchain_core.tools import tool

from ragas.integrations.langgraph import convert_to_ragas_messages

In [2]:
load_dotenv()

if not os.getenv("GOOGLE_API_KEY"):
    raise ValueError(
        "GOOGLE_API_KEY is missing. Please add it to the .env file."
    )

---
## 4. Domain Tool Definitions

We define three specialized travel tools decorated with LangChain's `@tool`:
1. **`search_flights`**: Returns available flight options with pricing and routes.
2. **`find_hotels`**: Finds hotel accommodations matching requested star ratings.
3. **`get_weather`**: Retrieves weather and temperature information for a destination.

In [3]:
@tool
def search_flights(origin: str, destination: str) -> str:
    """
    Search flights between an origin and destination city.
    """

    return f"""
Available flights from {origin} to {destination}:

1. Emirates EK569
   Price: ₹28,000
   Type: Non-stop

2. Air India AI995
   Price: ₹24,500
   Type: 1 stop
""".strip()

In [4]:
@tool
def find_hotels(city: str, stars: int) -> str:
    """
    Find hotels in a city based on star rating.
    """

    return f"""
Available {stars}-star hotels in {city}:

1. Novotel {city}
   Price: ₹8,500 per night

2. Hilton Garden Inn {city}
   Price: ₹9,200 per night
""".strip()

In [ ]:
@tool
def get_weather(city: str) -> str:
    """
    Get weather information for a city.
    """

    return f"""
Weather in {city}:

Temperature: 30°C
Condition: Sunny
""".strip()

---
## 5. LangChain Travel Planning Agent Construction with Google Gemini

We initialize `ChatGoogleGenerativeAI` using `gemini-3.1-flash-lite` with `temperature=0` to ensure deterministic, reproducible tool calling.

In [6]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0,
)

### 5.1 Interactive Model Sanity Check
A quick invocation to verify model connectivity and response generation.

In [7]:
llm.invoke("hi")

AIMessage(content=[{'type': 'text', 'text': 'Hello! How can I help you today?', 'extras': {'signature': 'EpwBCpkBAWkUfRPC32LLLHhB3wIQC95L3IHsk4/dXXijrQI11LNoh7rxhjIb+MOEJQlUYy2Qm1ZpC4BDNvon1wsvXaby6frrtOPKV2owfZRMw1ug9RCagDmLkFHfmiq8+a243SHMQDvEhQ8ydIqhXBg45j1o5qjQTykruQvPgEZawLgueprw3Xu09yaBQRuksmgI5BENoUvm0f7FXTW2'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1e5ed-a90d-7400-914a-9e2d5cfbc8f4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2, 'output_tokens': 9, 'total_tokens': 11, 'input_token_details': {'cache_read': 0}})

### 5.2 Agent Compilation with System Constraints
We compile the agent with `create_agent`, providing tool bindings and system rules to guide tool selection.

In [8]:
agent = create_agent(
    model=llm,

    tools=[
        search_flights,
        find_hotels,
        get_weather,
    ],

    system_prompt="""
You are a travel planning assistant.

Use the available tools whenever the user asks for
flight, hotel, or weather information.

Important rules:

1. Use search_flights for flight requests.
2. Use find_hotels for hotel requests.
3. Use get_weather only when weather is requested.
4. Do not call unnecessary tools.
5. Use city names exactly as provided by the user.
6. After using the tools, provide a clear final answer.
"""
)

### 5.3 Interactive Multi-Tool Sanity Check
We invoke the compiled agent with a travel request requiring multiple tool calls (flights + hotel).

In [10]:
agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": """
Plan a Dubai trip from Bangalore.
Find flights from Bangalore to Dubai
and find a 4-star hotel in Dubai.
""".strip(),
        }
    ]
})

{'messages': [HumanMessage(content='Plan a Dubai trip from Bangalore.\nFind flights from Bangalore to Dubai\nand find a 4-star hotel in Dubai.', additional_kwargs={}, response_metadata={}, id='74989b46-9392-4aab-a1bc-ffe10ef8d94a'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 55, 'prompt_tokens': 218, 'total_tokens': 273, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_936db6157f', 'id': 'chatcmpl-EkXMweqD9Uc1NdzKaPktaLzk1YCut', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1e009-4590-76ba-ab9b-d192c3e23243-0', tool_calls=[{'name': 'sea

---
# 🔬 PART I: RAGAS AGENTIC EVALUATION HARNESS

In this track, we evaluate the agent's multi-step decision trajectory using **Ragas**. 

Ragas evaluates agents by inspecting execution message traces:
1. Did the agent select the expected tools?
2. Did it invoke them with correct arguments?
3. Did the final trajectory satisfy the ground-truth reference goal?

## 6. Google GenAI Evaluator Client & Ragas Judge Setup

To ensure unbiased evaluation, the judge is decoupled from the agent. We initialize an independent Evaluator LLM using `gemini-3-flash-preview` via Ragas's **`llm_factory`** with `provider='google'`.

In [9]:
ragas_gemini_client = genai.Client(
    api_key=os.environ["GOOGLE_API_KEY"],
)

evaluator_llm = llm_factory(
    "gemini-3-flash-preview",
    provider="google",
    client=ragas_gemini_client,
)

## 7. Ragas Agentic Metrics Configuration

We instantiate three complementary agent evaluation metrics from Ragas:

In [10]:
tool_call_accuracy_metric = ToolCallAccuracy(strict_order=False)

###### Tool Call Accuracy → Checks whether the agent used the correct tool with the correct arguments.

In [11]:
tool_call_f1_metric = ToolCallF1()

###### Tool Call F1 → Measures the balance of precision and recall between expected and actual tool calls.

In [12]:
agent_goal_accuracy_metric = AgentGoalAccuracyWithReference(llm=evaluator_llm)

###### Agent Goal Accuracy → Checks whether the agent successfully completed the user’s final goal.

---
## 8. Ragas Golden Evaluation Dataset

We construct a multi-sample evaluation dataset. Each test case specifies:
* **`question`**: The user request prompt.
* **`reference_tool_calls`**: List of expected `ToolCall` instances with required arguments.
* **`reference_goal`**: Plain-text description of the expected final state.

In [13]:
dataset = [
    {
        "question": """
Plan a Dubai trip from Bangalore.
Find flights from Bangalore to Dubai
and find a 4-star hotel in Dubai.
""".strip(),

        "reference_tool_calls": [
            ToolCall(
                name="search_flights",
                args={
                    "origin": "Bangalore",
                    "destination": "Dubai",
                },
            ),
            ToolCall(
                name="find_hotels",
                args={
                    "city": "Dubai",
                    "stars": 4,
                },
            ),
        ],

        "reference_goal": """
The agent should provide flight options from Bangalore
to Dubai and 4-star hotel options in Dubai.
""".strip(),
    },

    {
        "question": """
Find flights from Bangalore to Singapore.
""".strip(),

        "reference_tool_calls": [
            ToolCall(
                name="search_flights",
                args={
                    "origin": "Bangalore",
                    "destination": "Singapore",
                },
            ),
        ],

        "reference_goal": """
The agent should provide flight options
from Bangalore to Singapore.
""".strip(),
    },

    {
        "question": """
Find a 5-star hotel in Mumbai.
""".strip(),

        "reference_tool_calls": [
            ToolCall(
                name="find_hotels",
                args={
                    "city": "Mumbai",
                    "stars": 5,
                },
            ),
        ],

        "reference_goal": """
The agent should provide 5-star hotel options
in Mumbai.
""".strip(),
    },
]

---
## 9. Agent Trace Extraction & Adapter Utilities

Helper utilities to extract text from messages, parse tool calls, and convert LangGraph execution states into Ragas message objects.

In [14]:
def get_message_text(message):

    content = message.content

    if isinstance(content, str):
        return content

    if isinstance(content, list):

        text_parts = []

        for block in content:

            if isinstance(block, str):
                text_parts.append(block)

            elif isinstance(block, dict) and "text" in block:
                text_parts.append(block["text"])

        return "\n".join(text_parts)

    return str(content)

In [15]:
def extract_tool_calls(ragas_trace):

    tool_calls = []

    for message in ragas_trace:
        message_tool_calls = getattr(
            message,
            "tool_calls",
            None,
        )

        if message_tool_calls:
            for tool_call in message_tool_calls:
                tool_calls.append(tool_call)

    return tool_calls

In [16]:
# async def run_agent(question):

#     result = await agent.ainvoke(
#         {
#             "messages": [
#                 {
#                     "role": "user",
#                     "content": question,
#                 }
#             ]
#         }
#     )

#     langchain_messages = result["messages"]

#     final_response = get_message_text(
#         langchain_messages[-1]
#     )

#     ragas_trace = convert_to_ragas_messages(
#         langchain_messages
#     )

#     actual_tool_calls = extract_tool_calls(
#         ragas_trace
#     )

#     return {
#         "response": final_response,
#         "ragas_trace": ragas_trace,
#         "actual_tool_calls": actual_tool_calls,
#     }

from ragas.messages import (
    HumanMessage as RagasHumanMessage,
    AIMessage as RagasAIMessage,
    ToolCall as RagasToolCall,
)


async def run_agent(question):

    result = await agent.ainvoke({
        "messages": [
            {
                "role": "user",
                "content": question,
            }
        ]
    })

    langchain_messages = result["messages"]

    final_response = get_message_text(
        langchain_messages[-1]
    )

    # ========================================================
    # EXTRACT TOOL CALLS DIRECTLY FROM LANGCHAIN MESSAGES
    # ========================================================

    actual_tool_calls = []

    for message in langchain_messages:

        tool_calls = getattr(
            message,
            "tool_calls",
            None,
        )

        if not tool_calls:
            continue

        for tool_call in tool_calls:

            actual_tool_calls.append(
                RagasToolCall(
                    name=tool_call["name"],
                    args=tool_call.get("args", {}),
                )
            )

    # ========================================================
    # CREATE RAGAS TRACE
    # ========================================================

    ragas_trace = [
        RagasHumanMessage(
            content=question,
        ),
        RagasAIMessage(
            content=final_response,
            tool_calls=actual_tool_calls,
        ),
    ]

    return {
        "response": final_response,
        "ragas_trace": ragas_trace,
        "actual_tool_calls": actual_tool_calls,
    }

In [17]:
# result = await agent.ainvoke({
#     "messages": [
#         {
#             "role": "user",
#             "content": """
# Plan a Dubai trip from Bangalore.
# Find flights from Bangalore to Dubai
# and find a 4-star hotel in Dubai.
# """.strip(),
#         }
#     ]
# })

# for message in result["messages"]:

#     print(type(message).__name__)

#     print(
#         "Tool Calls:",
#         getattr(message, "tool_calls", None),
#     )

#     print(
#         "Content:",
#         message.content,
#     )

#     print("-" * 100)

---
## 10. Single Test-Case Evaluation Pipeline

The function `evaluate_test_case(row)` runs the agent asynchronously, converts the trajectory into Ragas messages, measures all three metrics via `.ascore()`, and returns a consolidated dictionary.

In [18]:
async def evaluate_test_case(row):

    question = row["question"]
    reference_tool_calls = row["reference_tool_calls"]
    reference_goal = row["reference_goal"]

    # ========================================================
    # RUN AGENT
    # ========================================================

    agent_result = await run_agent(question)

    response = agent_result["response"]
    ragas_trace = agent_result["ragas_trace"]
    actual_tool_calls = agent_result["actual_tool_calls"]

    # ========================================================
    # TOOL CALL ACCURACY
    # ========================================================
    
    # Tool Call Accuracy → Checks whether the agent used the correct tool with the correct arguments.
    # Tool Call F1 → Measures the balance of precision and recall between expected and actual tool calls.
    # Agent Goal Accuracy → Checks whether the agent successfully completed the user’s final goal.
    
    tool_call_accuracy_result = await tool_call_accuracy_metric.ascore(
        user_input=ragas_trace,
        reference_tool_calls=reference_tool_calls,
    )

    # ========================================================
    # TOOL CALL F1
    # ========================================================

    tool_call_f1_result = await tool_call_f1_metric.ascore(
        user_input=ragas_trace,
        reference_tool_calls=reference_tool_calls,
    )

    # ========================================================
    # AGENT GOAL ACCURACY
    # ========================================================

    agent_goal_accuracy_result = await agent_goal_accuracy_metric.ascore(
        user_input=ragas_trace,
        reference=reference_goal,
    )

    # ========================================================
    # FORMAT TOOL CALLS
    # ========================================================

    actual_tool_calls_text = [
        f"{tool_call.name}({tool_call.args})"
        for tool_call in actual_tool_calls
    ]

    reference_tool_calls_text = [
        f"{tool_call.name}({tool_call.args})"
        for tool_call in reference_tool_calls
    ]

    # ========================================================
    # RETURN RESULT
    # ========================================================

    return {
        "question": question,
        "response": response,
        "actual_tool_calls": actual_tool_calls_text,
        "reference_tool_calls": reference_tool_calls_text,
        "tool_call_accuracy": tool_call_accuracy_result.value,
        "tool_call_f1": tool_call_f1_result.value,
        "agent_goal_accuracy": agent_goal_accuracy_result.value,
    }

---
## 11. Batch Evaluation Runner

The function `run_evaluation()` iterates through all test cases in the dataset sequentially.

In [19]:
async def run_evaluation():

    results = []

    for index, row in enumerate(dataset, start=1):

        print("\n" + "=" * 100)
        print(f"Running Test Case {index}/{len(dataset)}")
        print("=" * 100)

        result = await evaluate_test_case(row)

        results.append(result)

    return results

---
## 12. Main Orchestration & Diagnostic Reporting (Ragas)

The `main()` function handles the complete evaluation lifecycle:
1. Executes batch evaluation over the dataset.
2. Converts results into a pandas DataFrame.
3. Computes average scores across Tool Call Accuracy, Tool Call F1, and Agent Goal Accuracy.
4. Exports results to `ragas_agent_evaluation_results.csv`.
5. Prints detailed per-sample breakdowns of actual vs expected tool calls.

In [20]:
async def main():

    print("\n" + "=" * 100)
    print("RAGAS AGENTIC AI EVALUATION")
    print("=" * 100)

    # ========================================================
    # RUN EVALUATION
    # ========================================================

    results = await run_evaluation()

    # ========================================================
    # CONVERT TO DATAFRAME
    # ========================================================

    df = pd.DataFrame(results)

    # ========================================================
    # DETAILED RESULTS
    # ========================================================

    print("\n" + "=" * 100)
    print("DETAILED RESULTS")
    print("=" * 100)

    columns_to_show = [
        "question",
        "tool_call_accuracy",
        "tool_call_f1",
        "agent_goal_accuracy",
    ]

    print(df[columns_to_show].to_string(index=False))

    # ========================================================
    # SUMMARY
    # ========================================================

    average_tool_call_accuracy = df["tool_call_accuracy"].mean()
    average_tool_call_f1 = df["tool_call_f1"].mean()
    average_agent_goal_accuracy = df["agent_goal_accuracy"].mean()

    print("\n" + "=" * 100)
    print("SUMMARY")
    print("=" * 100)

    print(f"Average Tool Call Accuracy : {average_tool_call_accuracy:.2f}")
    print(f"Average Tool Call F1       : {average_tool_call_f1:.2f}")
    print(f"Average Agent Goal Accuracy: {average_agent_goal_accuracy:.2f}")

    # ========================================================
    # SAVE RESULTS
    # ========================================================

    output_file = "ragas_agent_evaluation_results.csv"

    df.to_csv(
        output_file,
        index=False,
    )

    print("\nResults saved to:")
    print(output_file)

    # ========================================================
    # PRINT EACH TEST CASE
    # ========================================================

    print("\n" + "=" * 100)
    print("DETAILED AGENT RESULTS")
    print("=" * 100)

    for _, result_row in df.iterrows():

        print("\nQUESTION:")
        print(result_row["question"])

        print("\nAGENT RESPONSE:")
        print(result_row["response"])

        print("\nEXPECTED TOOL CALLS:")
        print(result_row["reference_tool_calls"])

        print("\nACTUAL TOOL CALLS:")
        print(result_row["actual_tool_calls"])

        print("\nTOOL CALL ACCURACY:")
        print(result_row["tool_call_accuracy"])

        print("\nTOOL CALL F1:")
        print(result_row["tool_call_f1"])

        print("\nAGENT GOAL ACCURACY:")
        print(result_row["agent_goal_accuracy"])

        print("\n" + "-" * 100)

### 12.1 Execution Trigger (Ragas)
Executes `await main()` to run the Ragas benchmark and export results.

In [23]:
await main()


RAGAS AGENTIC AI EVALUATION

Running Test Case 1/3

Running Test Case 2/3

Running Test Case 3/3

DETAILED RESULTS
                                                                                                  question  tool_call_accuracy  tool_call_f1  agent_goal_accuracy
Plan a Dubai trip from Bangalore.\nFind flights from Bangalore to Dubai\nand find a 4-star hotel in Dubai.                 1.0           1.0                  1.0
                                                                 Find flights from Bangalore to Singapore.                 1.0           1.0                  1.0
                                                                            Find a 5-star hotel in Mumbai.                 1.0           1.0                  1.0

SUMMARY
Average Tool Call Accuracy : 1.00
Average Tool Call F1       : 1.00
Average Agent Goal Accuracy: 1.00

Results saved to:
ragas_agent_evaluation_results.csv

DETAILED AGENT RESULTS

QUESTION:
Plan a Dubai trip from Bangalore.
Fi

---
# 🧪 PART II: DEEPEVAL AGENTIC EVALUATION HARNESS

## Deepeval agent Evaluation

DeepEval models agent interactions through structured **Conversational Test Cases**:
* **`Turn`**: Represents an individual conversational exchange, capturing the user prompt, assistant response, and all associated `ToolCall` objects.
* **`ConversationalTestCase`**: Represents the end-to-end multi-turn trajectory container.

## 13. DeepEval Tool & Agent Re-declaration

We re-declare the domain tools and instantiate an isolated travel agent configured with `ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0)`.

In [ ]:
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool

from deepeval.models import GeminiModel

from deepeval.test_case import (
    LLMTestCase,
    ConversationalTestCase,
    Turn,
    ToolCall,
)

from deepeval.metrics import (
    ToolCorrectnessMetric,
    ArgumentCorrectnessMetric,
    GoalAccuracyMetric,
)

In [ ]:
@tool
def search_flights(origin: str, destination: str) -> str:
    """
    Search flights between an origin and destination city.
    """

    return f"""
Available flights from {origin} to {destination}:

1. Emirates EK569
   Price: ₹28,000
   Type: Non-stop

2. Air India AI995
   Price: ₹24,500
   Type: 1 stop
""".strip()


@tool
def find_hotels(city: str, stars: int) -> str:
    """
    Find hotels in a city based on star rating.
    """

    return f"""
Available {stars}-star hotels in {city}:

1. Novotel {city}
   Price: ₹8,500 per night

2. Hilton Garden Inn {city}
   Price: ₹9,200 per night
""".strip()


@tool
def get_weather(city: str) -> str:
    """
    Get weather information for a city.
    """

    return f"""
Weather in {city}:

Temperature: 30°C
Condition: Sunny
""".strip()

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0,
)

In [ ]:
agent = create_agent(
    model=llm,
    tools=[
        search_flights,
        find_hotels,
        get_weather,
    ],
    system_prompt="""
You are a travel planning assistant.

Use the available tools whenever the user asks for
flight, hotel, or weather information.

Rules:

1. Use search_flights for flight requests.
2. Use find_hotels for hotel requests.
3. Use get_weather only when weather is requested.
4. Do not call unnecessary tools.
5. Use city names exactly as provided by the user.
6. After using the tools, provide a clear final answer.
""",
)

---
## 14. DeepEval Evaluation Dataset with Expected Tool Calls

In DeepEval, expected tools are formally defined as **`ToolCall`** objects with `name` and `input_parameters` dictionaries.

In [ ]:
dataset = [
    {
        "question": """
Plan a Dubai trip from Bangalore.
Find flights from Bangalore to Dubai
and find a 4-star hotel in Dubai.
""".strip(),

        "expected_tools": [
            ToolCall(
                name="search_flights",
                input_parameters={
                    "origin": "Bangalore",
                    "destination": "Dubai",
                },
            ),
            ToolCall(
                name="find_hotels",
                input_parameters={
                    "city": "Dubai",
                    "stars": 4,
                },
            ),
        ],
    },

    {
        "question": """
Find flights from Bangalore to Singapore.
""".strip(),

        "expected_tools": [
            ToolCall(
                name="search_flights",
                input_parameters={
                    "origin": "Bangalore",
                    "destination": "Singapore",
                },
            ),
        ],
    },

    {
        "question": """
Find a 5-star hotel in Mumbai.
""".strip(),

        "expected_tools": [
            ToolCall(
                name="find_hotels",
                input_parameters={
                    "city": "Mumbai",
                    "stars": 5,
                },
            ),
        ],
    },
]

---
## 15. DeepEval Trace Extraction & Serialization Utilities

Helper functions to parse tool calls from message histories and serialize them into DeepEval `ToolCall` representations.

In [ ]:
def get_message_text(message):

    content = message.content

    if isinstance(content, str):
        return content

    if isinstance(content, list):

        text_parts = []

        for block in content:

            if isinstance(block, str):
                text_parts.append(block)

            elif isinstance(block, dict) and "text" in block:
                text_parts.append(block["text"])

        return "\n".join(text_parts)

    return str(content)

In [31]:
def extract_tool_calls(messages):

    tool_calls = []

    for message in messages:

        message_tool_calls = getattr(message, "tool_calls", None)

        if not message_tool_calls:
            continue

        for tool_call in message_tool_calls:

            tool_calls.append(
                ToolCall(
                    name=tool_call["name"],
                    input_parameters=tool_call.get("args", {}),
                )
            )

    return tool_calls

In [ ]:
def format_tool_calls(tool_calls):

    return [
        f"{tool_call.name}({tool_call.input_parameters})"
        for tool_call in tool_calls
    ]

---
## 16. DeepEval Agentic Metrics & Single Test-Case Pipeline

We configure three DeepEval metrics:
1. **`ToolCorrectnessMetric`**: Assesses whether the tools invoked match the expected tool names.
2. **`ArgumentCorrectnessMetric`**: Validates parameter names and values passed to each tool call.
3. **`GoalAccuracyMetric`**: LLM-as-a-Judge metric powered by **`GeminiModel(model="gemini-3-flash-preview")`** assessing conversational goal achievement.

The function `evaluate_test_case(row)` constructs a `Turn` with `actual_tool_calls`, wraps it in a `ConversationalTestCase`, and measures all three metrics.

In [ ]:
async def run_agent(question):

    result = await agent.ainvoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": question,
                }
            ]
        }
    )

    messages = result["messages"]

    response = get_message_text(messages[-1])

    actual_tool_calls = extract_tool_calls(messages)

    return {
        "response": response,
        "actual_tool_calls": actual_tool_calls,
    }

In [ ]:
async def evaluate_test_case(row):

    question = row["question"]
    expected_tools = row["expected_tools"]

    # ========================================================
    # RUN AGENT
    # ========================================================

    agent_result = await run_agent(question)

    response = agent_result["response"]
    actual_tool_calls = agent_result["actual_tool_calls"]

    # ========================================================
    # CREATE SINGLE-TURN TEST CASE
    # ========================================================

    agent_test_case = LLMTestCase(
        input=question,
        actual_output=response,
        tools_called=actual_tool_calls,
        expected_tools=expected_tools,
    )

    # ========================================================
    # CREATE TOOL CORRECTNESS METRIC
    # ========================================================

    tool_correctness_metric = ToolCorrectnessMetric(
        threshold=0.7,
        include_reason=True,
    )

    # ========================================================
    # CREATE ARGUMENT CORRECTNESS METRIC
    # ========================================================

    argument_correctness_metric = ArgumentCorrectnessMetric(
        threshold=0.7,
        model="gpt-5.4",
        include_reason=True,
        async_mode=False,
    )

    # ========================================================
    # TOOL CORRECTNESS EVALUATION
    # ========================================================

    tool_correctness_metric.measure(agent_test_case)

    # ========================================================
    # ARGUMENT CORRECTNESS EVALUATION
    # ========================================================

    argument_correctness_metric.measure(agent_test_case)

    # ========================================================
    # CREATE CONVERSATIONAL TEST CASE
    # ========================================================

    conversation_test_case = ConversationalTestCase(
        turns=[
            Turn(
                role="user",
                content=question,
            ),
            Turn(
                role="assistant",
                content=response,
                tools_called=actual_tool_calls,
            ),
        ]
    )

    # ========================================================
    # CREATE GOAL ACCURACY METRIC
    # ========================================================

    judge_model = GeminiModel(
        model="gemini-3-flash-preview",
        api_key=os.environ["GOOGLE_API_KEY"],
    )

    goal_accuracy_metric = GoalAccuracyMetric(
        threshold=0.7,
        model=judge_model,
        include_reason=True,
        async_mode=False,
    )

    # ========================================================
    # GOAL ACCURACY EVALUATION
    # ========================================================

    goal_accuracy_metric.measure(conversation_test_case)

    # ========================================================
    # RETURN RESULT
    # ========================================================

    return {
        "question": question,
        "response": response,
        "expected_tools": format_tool_calls(expected_tools),
        "actual_tool_calls": format_tool_calls(actual_tool_calls),

        "tool_correctness": tool_correctness_metric.score,
        "tool_correctness_reason": tool_correctness_metric.reason,

        "argument_correctness": argument_correctness_metric.score,
        "argument_correctness_reason": argument_correctness_metric.reason,

        "goal_accuracy": goal_accuracy_metric.score,
        "goal_accuracy_reason": goal_accuracy_metric.reason,
    }

---
## 17. DeepEval Batch Evaluation Runner

The function `run_evaluation()` sequentially evaluates all test cases in the dataset.

In [35]:
async def run_evaluation():

    results = []

    for index, row in enumerate(dataset, start=1):

        print("\n" + "=" * 100)
        print(f"Running Test Case {index}/{len(dataset)}")
        print("=" * 100)

        result = await evaluate_test_case(row)

        results.append(result)

    return results

---
## 18. DeepEval Main Orchestration & Diagnostic Reporting

The `main()` function:
1. Executes batch evaluation across all test cases.
2. Converts outputs into a pandas DataFrame.
3. Computes mean scores across Tool Correctness, Argument Correctness, and Goal Accuracy.
4. Exports the benchmark report to `deepeval_agent_evaluation_results.csv`.
5. Prints comprehensive qualitative rationales from the judge for every test case.

In [36]:
async def main():

    print("\n" + "=" * 100)
    print("DEEPEVAL AGENTIC AI EVALUATION")
    print("=" * 100)

    # ========================================================
    # RUN EVALUATION
    # ========================================================

    results = await run_evaluation()

    # ========================================================
    # CONVERT TO PANDAS
    # ========================================================

    df = pd.DataFrame(results)

    # ========================================================
    # DETAILED RESULTS
    # ========================================================

    print("\n" + "=" * 100)
    print("DETAILED RESULTS")
    print("=" * 100)

    columns_to_show = [
        "question",
        "tool_correctness",
        "argument_correctness",
        "goal_accuracy",
    ]

    print(df[columns_to_show].to_string(index=False))

    # ========================================================
    # CALCULATE AVERAGE SCORES
    # ========================================================

    average_tool_correctness = df["tool_correctness"].mean()
    average_argument_correctness = df["argument_correctness"].mean()
    average_goal_accuracy = df["goal_accuracy"].mean()

    # ========================================================
    # SUMMARY
    # ========================================================

    print("\n" + "=" * 100)
    print("SUMMARY")
    print("=" * 100)

    print(f"Average Tool Correctness     : {average_tool_correctness:.2f}")
    print(f"Average Argument Correctness : {average_argument_correctness:.2f}")
    print(f"Average Goal Accuracy        : {average_goal_accuracy:.2f}")

    # ========================================================
    # SAVE RESULTS
    # ========================================================

    output_file = "deepeval_agent_evaluation_results.csv"

    df.to_csv(output_file, index=False)

    print("\nResults saved to:")
    print(output_file)

    # ========================================================
    # PRINT DETAILED RESULTS
    # ========================================================

    print("\n" + "=" * 100)
    print("DETAILED AGENT EVALUATION REASONS")
    print("=" * 100)

    for _, result_row in df.iterrows():

        print("\nQUESTION:")
        print(result_row["question"])

        print("\nAGENT RESPONSE:")
        print(result_row["response"])

        print("\nEXPECTED TOOLS:")
        print(result_row["expected_tools"])

        print("\nACTUAL TOOL CALLS:")
        print(result_row["actual_tool_calls"])

        print("\nTOOL CORRECTNESS:")
        print(result_row["tool_correctness"])

        print("\nTOOL CORRECTNESS REASON:")
        print(result_row["tool_correctness_reason"])

        print("\nARGUMENT CORRECTNESS:")
        print(result_row["argument_correctness"])

        print("\nARGUMENT CORRECTNESS REASON:")
        print(result_row["argument_correctness_reason"])

        print("\nGOAL ACCURACY:")
        print(result_row["goal_accuracy"])

        print("\nGOAL ACCURACY REASON:")
        print(result_row["goal_accuracy_reason"])

        print("\n" + "-" * 100)

### 18.1 Execution Trigger (DeepEval)
Executes `await main()` to measure conversational agent metrics and export results.

In [37]:
await main()


DEEPEVAL AGENTIC AI EVALUATION

Running Test Case 1/3


d:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\env\Lib\site-packages\rich\live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')


Running Test Case 2/3


d:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\env\Lib\site-packages\rich\live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')


Running Test Case 3/3


d:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\env\Lib\site-packages\rich\live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')


DETAILED RESULTS
                                                                                                  question  tool_correctness  argument_correctness  goal_accuracy
Plan a Dubai trip from Bangalore.\nFind flights from Bangalore to Dubai\nand find a 4-star hotel in Dubai.               1.0                   1.0          0.625
                                                                 Find flights from Bangalore to Singapore.               1.0                   1.0          0.500
                                                                            Find a 5-star hotel in Mumbai.               1.0                   1.0          0.625

SUMMARY
Average Tool Correctness     : 1.00
Average Argument Correctness : 1.00
Average Goal Accuracy        : 0.58

Results saved to:
deepeval_agent_evaluation_results.csv

DETAILED AGENT EVALUATION REASONS

QUESTION:
Plan a Dubai trip from Bangalore.
Find flights from Bangalore to Dubai
and find a 4-star hotel in Dubai.

AGENT RE

---
## 19. Comprehensive Comparative Analysis: Ragas vs DeepEval for Agent Evaluation

Both **Ragas** and **DeepEval** offer powerful capabilities for agent testing, but they approach the problem through distinct abstractions:

| Evaluation Criterion | Ragas Agentic Evaluation | DeepEval Agentic Evaluation |
| :--- | :--- | :--- |
| **Trace Model** | Message state lists via `convert_to_ragas_messages` | Hierarchical `Turn` and `ConversationalTestCase` |
| **Tool Selection Metric** | `ToolCallAccuracy` (order agnostic or strict) | `ToolCorrectnessMetric` |
| **Argument Validation** | Built into `ToolCallAccuracy` | Dedicated `ArgumentCorrectnessMetric` |
| **Tool Precision & Recall**| `ToolCallF1` (harmonic mean) | Evaluated through per-tool matching |
| **Goal Attainment** | `AgentGoalAccuracyWithReference` | `GoalAccuracyMetric` |
| **Judge LLM Interface** | `llm_factory(provider='google')` via Google GenAI | Native `GeminiModel` class wrapper |
| **Best Used For** | Continuous benchmarking, trajectory tracing, research | Conversational unit testing, CI/CD Pytest assertions |

---

### 💡 Production Best Practices & Key Takeaways:
1. **Combine Rule-Based & LLM Metrics**: Use deterministic metrics (`ToolCorrectness`, `ToolCallF1`) for fast, zero-cost CI checks, and LLM-as-a-Judge (`GoalAccuracy`) for semantic goal completion.
2. **Keep Temperature at 0**: When running agents in production or evaluation benchmarks, set `temperature=0` to ensure repeatable tool calling.
3. **Log Full Trajectories**: Retain intermediate reasoning steps and tool arguments in CSV reports to debug tool hallucination and prompt drift.